In [50]:
import json
import networkx as nx
import pandas as pd
import numpy as np
from node2vec import Node2Vec
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# GET DATA

In [4]:
with open("../data/simulation_data/run_7/run_7/trajectories/all_trajectories.json", "r") as f:
    data = json.load(f)

In [5]:
all_trajs = []
for agent_id, trajs in data.items():
    for traj in trajs:
        path = traj["path"]
        goal = traj["goal_node"]
        hour = traj["hour"]
        all_trajs.append({"path": path, "goal": goal, "hour": hour})

print(f"Loaded {len(all_trajs)} trajectories total.")

Loaded 100000 trajectories total.


In [36]:
# all_trajs

# [
#     {
#         'path': ['nodeA', 'nodeB', 'nodeC', ..., 'nodeN'],
#         'goal': ['nodeZ'],
#         'hour': 1
#     }
#     {
#         'path': ['nodeA', 'nodeB', 'nodeC', ..., 'nodeN'],
#         'goal': ['nodeZ'],
#         'hour': 1
#     }
#     ...
#     {
#         'path': ['nodeA', 'nodeB', 'nodeC', ..., 'nodeN'],
#         'goal': ['nodeZ'],
#         'hour': 1
#     }
# ]

# CREATE NODE2VEC OF UCSD GRAPH

In [9]:
G = nx.read_graphml("../data/processed/ucsd_walk_full.graphml")
print(f"Graph type: {type(G)}")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Is directed: {G.is_directed()}")

Graph type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Number of nodes: 2823
Number of edges: 7390
Is directed: True


In [ ]:
# node2vec = Node2Vec(
#     G,
#     dimensions=64,      # size of the embedding vector for each node
#     walk_length=20,     # length of each random walk
#     num_walks=100,      # number of walks per node
#     workers=4,          # CPU cores (adjust for your machine)
#     quiet=False         # shows progress
# )

Computing transition probabilities:   0%|          | 0/2823 [00:00<?, ?it/s]

In [ ]:
# model = node2vec.fit(
#     window=10,         # context window size (like Word2Vec)
#     min_count=1,       # keep all nodes
#     batch_words=4
# )

In [ ]:
# model.wv.save_word2vec_format("../data/processed/ucsd_node2vec.emb")

In [ ]:
emb_path = "../data/processed/ucsd_node2vec.emb"
node_embeddings = {}

with open(emb_path, "r") as f:
    header = f.readline()
    for line in f:
        parts = line.strip().split()
        node_id = parts[0]
        vector = np.array(parts[1:], dtype=float)
        node_embeddings[node_id] = vector

print(f"✅ Loaded {len(node_embeddings)} node embeddings, "
    f"each of dimension {len(next(iter(node_embeddings.values())))}")

✅ Loaded 2823 node embeddings, each of dimension 64


In [21]:
# node_embeddings

# ONE HOT ENCODE HOURS

In [ ]:
hours = np.array([[traj["hour"]] for traj in all_trajs])
hour_encoder = OneHotEncoder(categories='auto', sparse_output=False)
hour_onehot = hour_encoder.fit_transform(hours)

print(hour_onehot.shape)  # (num_trajs, 21), assuming that there were only 21 distinct hours in the simulation data

(100000, 21)


In [ ]:
# basically 10000 rows of 21days, because 10000 trajs

# [[0, 1, ..., 0, 0],
#  [0, 0, ..., 0, 0],
#  [1, 0, ..., 0, 0],
#  ...,
#  [0, 0, ..., 0, 0]]

# CREATE X and y (ADD EMBEDDED NODES FIRST)

In [ ]:
X = []  # list of sequences
y = []  # list of goal nodes

for traj in all_trajs:
    path = traj["path"]
    goal = traj["goal"]
    hour = traj["hour"]

    # loop through each node in a path. if possible, convert that string node into the node_embedded vector
    seq = []
    for node in path:
        if node in node_embeddings:
            seq.append(node_embeddings[node])
        else:
            continue

    seq = np.array(seq)
    X.append(seq)
    y.append(goal)

In [27]:
# Trajectory A: [A, B, C, D]
# Trajectory B: [X, Y]

# After padding (max length = 4):

# A: [A, B, C, D]
# B: [X, Y, 0, 0]   ← zeros added (padding)

max_len = max(len(seq) for seq in X)
embedding_dim = X[0].shape[1]

X_padded = np.zeros((len(X), max_len, embedding_dim))

for i, seq in enumerate(X):
    length = len(seq)
    X_padded[i, :length, :] = seq

In [30]:
print(X_padded.shape)
print(len(y))

(100000, 107, 64)
100000


In [ ]:
# [
#     [
#         'path': [[num1, num2, ..., num64], [num1, num2, ... num64], 'node3', ..., 'node107'],
#     ]
#     [
#         'path': ['node1', 'node2', 'node3', ..., 'node107'],
#     ]
#     ... x 100000
#     [
#         'path': ['node1', 'node2', 'node3', ..., 'node107'],
#     ]
# ]

# y = [endnode1, endnode2, ..., endnode100000]

In [ ]:
i = 0
print("Trajectory index:", i)
print("Sequence shape:", X_padded[i].shape)
print("Goal:", y[i])

# visualize first few embedding vectors
print("First 2 node embeddings of path:\n", X_padded[i][:2])

# basically, its like this [[node1], [node2], ...]

Trajectory index: 0
Sequence shape: (107, 64)
Goal: 10293853345
First 2 node embeddings of path:
 [[ 0.19072017 -0.05328501  0.85272646  0.69022685 -0.2506782  -0.20613174
   0.23331104 -0.52978945 -0.2141229  -0.06948136  0.27901968 -0.14948547
   0.70635384 -0.7256938   0.56325585 -0.09922434 -1.0660905  -0.5505161
  -1.1617543   0.8479003  -0.402322   -0.73409593 -0.3524349  -1.0138953
  -0.39000052 -0.04378304 -1.429783    1.2644799  -0.4536941   0.33104482
  -0.01283199  0.13573734  0.3016323   0.8375865   0.9726326   0.52279
   0.03023112  0.32270414 -1.3524868   0.28140256  0.465262    0.00228757
  -0.0707736  -0.11776069  0.1572885  -0.3093049  -0.37073    -0.42094594
  -0.38965636 -0.22815648  1.0269585  -0.31597942 -0.34763452 -1.0365183
   0.10998794 -0.09831373 -0.08979154 -0.14540347  0.30425596  0.8798612
  -0.3772001  -1.4737209   0.07870285 -0.6300602 ]
 [ 0.11331052 -0.12565514  0.7622888   0.6357405  -0.11702609 -0.1949537
   0.10250951 -0.4211418  -0.1751653  -0.0942

# NOW APPEND ONE HOT ENCODED FEATURE TO X

In [ ]:
# Suppose:
# X_padded.shape = (num_trajs, max_len, embedding_dim)
# hour_onehot.shape = (num_trajs, hour_dim)  <- from OneHotEncoder

num_trajs, max_len, embedding_dim = X_padded.shape
hour_dim = hour_onehot.shape[1]
hours_expanded = np.repeat(hour_onehot[:, np.newaxis, :], max_len, axis=1)
X_final = np.concatenate([X_padded, hours_expanded], axis=-1)
# shape -> (num_trajs, max_len, embedding_dim + hour_dim)

print("Final X shape:", X_final.shape)
# (100000, 107, 64+21)

Final X shape: (100000, 107, 85)


# PREPARE y

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # Converts goal_node IDs -> integers
num_classes = len(le.classes_)

print("Number of unique goal nodes:", num_classes)
# array([ 25,  25,  25, ...,  80,  38, 208], dtype=int64)

Number of unique goal nodes: 230


In [ ]:
# np.savez_compressed("traj_data.npz", X=X_padded, y=y, hours=hour_onehot)
